# Fase 11 — Investigación en profundidad: clasificación, calendario y diagnóstico de Afición

**Requisito previo:** base de datos `golazo_growup` cargada (`python -m src.cargar_datos`).

Convención usada en todo el notebook: los **'días fuertes de audiencia'** son los 2 días de la semana con más vistas medias en `evolucion_diaria` (30 días reales). Es una muestra pequeña (~4 observaciones por día de la semana) — se trata como el mejor proxy disponible, no como un patrón verificado a largo plazo.

## 0. Carga de datos y columnas derivadas

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

from src import db_connection as dbc

url = dbc.DATABASE_URL or (
    f"postgresql+psycopg2://{dbc.DB_CONFIG['user']}:{dbc.DB_CONFIG['password']}"
    f"@{dbc.DB_CONFIG['host']}:{dbc.DB_CONFIG['port']}/{dbc.DB_CONFIG['dbname']}"
)
if url.startswith('postgresql://'):
    url = url.replace('postgresql://', 'postgresql+psycopg2://', 1)
engine = create_engine(url)

video = pd.read_sql('SELECT * FROM video', engine, parse_dates=['fecha_publicacion'])
retencion = pd.read_sql('SELECT * FROM retencion_audiencia', engine)
evolucion = pd.read_sql('SELECT * FROM evolucion_diaria', engine, parse_dates=['fecha'])

ORDEN_DIAS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
HOY = pd.Timestamp.now().normalize()

evolucion['dia_semana'] = evolucion['fecha'].dt.day_name()
evolucion['suscriptores_netos'] = evolucion['subscribers_gained'] - evolucion['subscribers_lost']

UMBRAL_SHORT_SEGUNDOS = 180
video['es_short'] = video['duracion_segundos'] <= UMBRAL_SHORT_SEGUNDOS
video['formato'] = video['es_short'].map({True: 'Short', False: 'Largo'})
video['dia_semana_publicacion'] = video['fecha_publicacion'].dt.day_name()

# Antigüedad: se recalcula en cada ejecución contra HOY, así que se actualiza sola día a día
video['antiguedad_dias'] = (HOY - video['fecha_publicacion']).dt.days.clip(lower=1)
video['views_por_dia'] = video['views_totales'] / video['antiguedad_dias']

# Umbral mínimo de antigüedad para usar 'views_por_dia' de forma fiable: un vídeo con 1-2 días
# de vida puede dar ratios absurdos (denominador casi cero) que no reflejan una velocidad real,
# solo ruido de publicación reciente. Se exige al menos 7 días antes de fiarse de su velocidad.
MIN_DIAS_PARA_VELOCIDAD = 7
video['apto_velocidad'] = video['antiguedad_dias'] >= MIN_DIAS_PARA_VELOCIDAD

video['ratio_likes_vista'] = video['likes'] / video['views_totales']
video['ratio_comentarios_vista'] = video['comentarios'] / video['views_totales']
video['z_views_categoria_formato'] = video.groupby(['categoria', 'formato'])['views_totales'].transform(
    lambda s: (s - s.mean()) / s.std() if s.std() > 0 else 0
)

retencion_media_por_video = retencion.groupby('video_id')['audience_watch_ratio'].mean().rename('retencion_media')
video = video.merge(retencion_media_por_video, on='video_id', how='left')

patron_semanal_audiencia = evolucion.groupby('dia_semana')['views'].mean().reindex(ORDEN_DIAS)
dias_fuertes = patron_semanal_audiencia.sort_values(ascending=False).head(2).index.tolist()
video['es_dia_fuerte'] = video['dia_semana_publicacion'].isin(dias_fuertes)

print(f'HOY (referencia de antigüedad): {HOY.date()}')
print(f'Días fuertes de audiencia: {dias_fuertes}')
print(f'Vídeos: {len(video)} | Rango de antigüedad: {video["antiguedad_dias"].min()} a {video["antiguedad_dias"].max()} días')

HOY (referencia de antigüedad): 2026-09-14
Días fuertes de audiencia: ['Saturday', 'Sunday']
Vídeos: 338 | Rango de antigüedad: 1 a 4161 días


## 1. Clasificación de vídeos por vistas y tiempo publicado

Por cada vídeo: fecha y día de publicación, antigüedad en días (recalculada contra la fecha de hoy cada vez que se ejecuta el notebook), nivel de vistas, y una métrica de velocidad (`views_por_dia`) que separa 'tiene muchas vistas' de 'ha conseguido muchas vistas rápido'.

In [2]:
video['nivel_views'] = pd.qcut(video['views_totales'], q=4, labels=['Bajo', 'Medio', 'Alto', 'Muy alto'])
print('Vídeos por nivel de vistas:')
display(video['nivel_views'].value_counts().reindex(['Bajo', 'Medio', 'Alto', 'Muy alto']))

columnas_ficha = ['titulo', 'categoria', 'formato', 'fecha_publicacion', 'dia_semana_publicacion',
                   'antiguedad_dias', 'views_totales', 'views_por_dia', 'likes', 'retencion_media', 'nivel_views']

Vídeos por nivel de vistas:


nivel_views
Bajo        85
Medio       84
Alto        84
Muy alto    85
Name: count, dtype: int64

In [3]:
print('TOP 10 por VISTAS TOTALES (acumulado de toda la vida del vídeo):')
display(video.sort_values('views_totales', ascending=False).head(10)[columnas_ficha].round(2))

TOP 10 por VISTAS TOTALES (acumulado de toda la vida del vídeo):


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10960\3696186135.py:2: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(video.sort_values('views_totales', ascending=False).head(10)[columnas_ficha].round(2))


,titulo,categoria,formato,fecha_publicacion,dia_semana_publicacion,antiguedad_dias,views_totales,views_por_dia,likes,retencion_media,nivel_views
84,Análisis del partido - Jornada 85,Seguimiento del club,Short,2018-02-22,Thursday,3126,20091,6.43,636,0.37,Muy alto
111,Nuestra opinión tras el partido #112,Opinión post-partido,Short,2019-01-21,Monday,2793,20016,7.17,688,0.44,Muy alto
168,Análisis del partido - Jornada 169,Seguimiento del club,Largo,2020-12-24,Thursday,2090,19849,9.50,668,0.43,Muy alto
0,Nuestra opinión tras el partido #1,Opinión post-partido,Short,2015-04-24,Friday,4161,19844,4.77,650,0.44,Muy alto
79,Nuestra opinión tras el partido #80,Opinión post-partido,Short,2017-12-21,Thursday,3189,18822,5.90,661,0.43,Muy alto
318,Análisis del partido - Jornada 319,Seguimiento del club,Short,2026-01-18,Sunday,239,18768,78.53,656,0.33,Muy alto
54,Análisis del partido - Jornada 55,Seguimiento del club,Largo,2017-02-19,Sunday,3494,18505,5.30,544,0.41,Muy alto
173,Análisis del partido - Jornada 174,Seguimiento del club,Largo,2021-02-28,Sunday,2024,18388,9.08,522,0.37,Muy alto
198,Análisis del partido - Jornada 199,Seguimiento del club,Largo,2022-01-04,Tuesday,1714,18378,10.72,572,0.42,Muy alto
142,Análisis del partido - Jornada 143,Seguimiento del club,Largo,2020-02-07,Friday,2411,18313,7.60,504,0.37,Muy alto


In [4]:
print('TOP 10 por RETENCIÓN MEDIA:')
display(video.sort_values('retencion_media', ascending=False).head(10)[columnas_ficha].round(2))

TOP 10 por RETENCIÓN MEDIA:


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10960\264988412.py:2: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(video.sort_values('retencion_media', ascending=False).head(10)[columnas_ficha].round(2))


,titulo,categoria,formato,fecha_publicacion,dia_semana_publicacion,antiguedad_dias,views_totales,views_por_dia,likes,retencion_media,nivel_views
154,Análisis del partido - Jornada 155,Seguimiento del club,Largo,2020-07-03,Friday,2264,1732,0.77,56,0.45,Medio
304,Nuestra opinión tras el partido #305,Opinión post-partido,Short,2025-07-29,Tuesday,412,16840,40.87,813,0.45,Muy alto
6,Análisis del partido - Jornada 7,Seguimiento del club,Largo,2015-07-09,Thursday,4085,9825,2.41,420,0.45,Muy alto
166,Análisis del partido - Jornada 167,Seguimiento del club,Largo,2020-11-29,Sunday,2115,493,0.23,15,0.45,Bajo
182,Curiosidad futbolística #183,Curiosidades de fútbol,Largo,2021-06-20,Sunday,1912,2143,1.12,94,0.45,Medio
51,Nuestra opinión tras el partido #52,Opinión post-partido,Largo,2017-01-09,Monday,3535,11819,3.34,461,0.45,Muy alto
281,Análisis del partido - Jornada 282,Seguimiento del club,Short,2024-10-19,Saturday,695,1574,2.26,45,0.45,Medio
170,Análisis del partido - Jornada 171,Seguimiento del club,Largo,2021-01-23,Saturday,2060,979,0.48,45,0.45,Bajo
174,Rumores de fichajes: novedad #175,Fichajes,Short,2021-03-12,Friday,2012,14424,7.17,586,0.44,Muy alto
139,Nuestra opinión tras el partido #140,Opinión post-partido,Largo,2019-12-31,Tuesday,2449,13639,5.57,608,0.44,Muy alto


In [5]:
print('TOP 10 por LIKES:')
display(video.sort_values('likes', ascending=False).head(10)[columnas_ficha].round(2))

TOP 10 por LIKES:


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10960\155872550.py:2: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(video.sort_values('likes', ascending=False).head(10)[columnas_ficha].round(2))


,titulo,categoria,formato,fecha_publicacion,dia_semana_publicacion,antiguedad_dias,views_totales,views_por_dia,likes,retencion_media,nivel_views
81,Análisis del partido - Jornada 82,Seguimiento del club,Short,2018-01-15,Monday,3164,17067,5.39,819,0.37,Muy alto
304,Nuestra opinión tras el partido #305,Opinión post-partido,Short,2025-07-29,Tuesday,412,16840,40.87,813,0.45,Muy alto
45,Análisis del partido - Jornada 46,Seguimiento del club,Largo,2016-10-27,Thursday,3609,17127,4.75,812,0.38,Muy alto
260,Análisis del partido - Jornada 261,Seguimiento del club,Largo,2024-02-08,Thursday,949,16776,17.68,771,0.35,Muy alto
292,Curiosidad futbolística #293,Curiosidades de fútbol,Largo,2025-03-07,Friday,556,18125,32.60,747,0.43,Muy alto
258,Rumores de fichajes: novedad #259,Fichajes,Largo,2024-01-09,Tuesday,979,17967,18.35,744,0.40,Muy alto
225,Análisis del partido - Jornada 226,Seguimiento del club,Short,2022-11-30,Wednesday,1384,16025,11.58,742,0.43,Muy alto
324,Curiosidad futbolística #325,Curiosidades de fútbol,Largo,2026-04-01,Wednesday,166,15756,94.92,720,0.35,Muy alto
57,Análisis del partido - Jornada 58,Seguimiento del club,Largo,2017-03-28,Tuesday,3457,14511,4.20,700,0.34,Muy alto
119,Nuestra opinión tras el partido #120,Opinión post-partido,Largo,2019-04-28,Sunday,2696,16551,6.14,699,0.37,Muy alto


**Nota metodológica:** para el ranking de velocidad se exige un mínimo de 7 días de antigüedad — sin este filtro, un vídeo publicado ayer con pocas vistas puede dar un ratio absurdamente alto solo porque el denominador (días) es casi cero, no porque esté funcionando especialmente bien.

In [6]:
print('TOP 10 por VELOCIDAD (views_por_dia, solo vídeos con 7+ dias de antiguedad para evitar outliers):')
top10_velocidad = video[video['apto_velocidad']].sort_values('views_por_dia', ascending=False).head(10)
display(top10_velocidad[columnas_ficha].round(2))

TOP 10 por VELOCIDAD (views_por_dia, solo vídeos con 7+ dias de antiguedad para evitar outliers):


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10960\1563334332.py:3: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(top10_velocidad[columnas_ficha].round(2))


,titulo,categoria,formato,fecha_publicacion,dia_semana_publicacion,antiguedad_dias,views_totales,views_por_dia,likes,retencion_media,nivel_views
336,Nuestra opinión tras el partido #337,Opinión post-partido,Largo,2026-08-29,Saturday,16,14242,890.12,542,0.34,Muy alto
334,Curiosidad futbolística #335,Curiosidades de fútbol,Short,2026-08-09,Sunday,36,13296,369.33,356,0.36,Muy alto
324,Curiosidad futbolística #325,Curiosidades de fútbol,Largo,2026-04-01,Wednesday,166,15756,94.92,720,0.35,Muy alto
318,Análisis del partido - Jornada 319,Seguimiento del club,Short,2026-01-18,Sunday,239,18768,78.53,656,0.33,Muy alto
332,Análisis del partido - Jornada 333,Seguimiento del club,Largo,2026-07-15,Wednesday,61,3266,53.54,134,0.35,Medio
331,Nuestra opinión tras el partido #332,Opinión post-partido,Short,2026-06-29,Monday,77,3494,45.38,150,0.41,Medio
326,Rumores de fichajes: novedad #327,Fichajes,Largo,2026-04-28,Tuesday,139,5887,42.35,189,0.41,Alto
304,Nuestra opinión tras el partido #305,Opinión post-partido,Short,2025-07-29,Tuesday,412,16840,40.87,813,0.45,Muy alto
309,Nuestra opinión tras el partido #310,Opinión post-partido,Largo,2025-10-04,Saturday,345,12069,34.98,432,0.41,Muy alto
299,Análisis del partido - Jornada 300,Seguimiento del club,Short,2025-06-03,Tuesday,468,16240,34.70,526,0.38,Muy alto


In [7]:
# Comparación clave: ¿los líderes por vistas totales son los mismos que los líderes por velocidad?
top10_views = set(video.sort_values('views_totales', ascending=False).head(10)['video_id'])
top10_vel = set(top10_velocidad['video_id'])
interseccion = top10_views & top10_vel
print(f'Vídeos que están en AMBOS Top 10 (vistas totales Y velocidad): {len(interseccion)} de 10')

antiguedad_top_views = video[video['video_id'].isin(top10_views)]['antiguedad_dias'].mean()
antiguedad_top_vel = video[video['video_id'].isin(top10_vel)]['antiguedad_dias'].mean()
print(f"Antigüedad media del Top 10 por vistas totales: {antiguedad_top_views:.0f} días")
print(f"Antigüedad media del Top 10 por velocidad (ya filtrado >= 7 días): {antiguedad_top_vel:.0f} días")
if antiguedad_top_views > antiguedad_top_vel * 1.5:
    print('\nLECTURA: el Top de vistas totales está dominado por vídeos antiguos que simplemente han tenido '
          'más tiempo para acumular — el Top de velocidad es la referencia más honesta de qué está '
          'funcionando BIEN AHORA, no solo qué es viejo.')

Vídeos que están en AMBOS Top 10 (vistas totales Y velocidad): 1 de 10
Antigüedad media del Top 10 por vistas totales: 2524 días
Antigüedad media del Top 10 por velocidad (ya filtrado >= 7 días): 196 días

LECTURA: el Top de vistas totales está dominado por vídeos antiguos que simplemente han tenido más tiempo para acumular — el Top de velocidad es la referencia más honesta de qué está funcionando BIEN AHORA, no solo qué es viejo.


## 2. Calendario: Opinión post-partido (Shorts) y Seguimiento del club (Largo)

Qué día de la semana se publica cada combinación categoría+formato hoy, y qué día habría que tenerlos listos para llegar publicados ANTES de los días fuertes de audiencia.

In [8]:
opinion_shorts = video[(video['categoria'] == 'Opinión post-partido') & (video['formato'] == 'Short')]
seguimiento_largos = video[(video['categoria'] == 'Seguimiento del club') & (video['formato'] == 'Largo')]

print(f'Opinión post-partido en Shorts: {len(opinion_shorts)} vídeos')
dist_opinion = opinion_shorts['dia_semana_publicacion'].value_counts(normalize=True).mul(100).reindex(ORDEN_DIAS).fillna(0)
display(dist_opinion.round(1))

print(f'\nSeguimiento del club en Largo: {len(seguimiento_largos)} vídeos')
dist_seguimiento = seguimiento_largos['dia_semana_publicacion'].value_counts(normalize=True).mul(100).reindex(ORDEN_DIAS).fillna(0)
display(dist_seguimiento.round(1))

Opinión post-partido en Shorts: 22 vídeos


dia_semana_publicacion
Monday       18.2
Tuesday      22.7
Wednesday    13.6
Thursday      9.1
Friday       13.6
Saturday      9.1
Sunday       13.6
Name: proportion, dtype: float64


Seguimiento del club en Largo: 120 vídeos


dia_semana_publicacion
Monday       10.8
Tuesday      14.2
Wednesday    12.5
Thursday     15.0
Friday       17.5
Saturday     14.2
Sunday       15.8
Name: proportion, dtype: float64

In [9]:
# Recomendación de calendario: publicar 1-2 días ANTES del día fuerte más próximo,
# para que el vídeo ya esté ganando tracción cuando llegue el pico de audiencia.
indice_dias = {d: i for i, d in enumerate(ORDEN_DIAS)}

for nombre, dist in [('Opinión post-partido (Short)', dist_opinion), ('Seguimiento del club (Largo)', dist_seguimiento)]:
    dia_actual_top = dist.idxmax()
    print(f"\n{nombre}: día actual más usado para publicar -> {dia_actual_top}")
    for dia_fuerte in dias_fuertes:
        idx_fuerte = indice_dias[dia_fuerte]
        dia_recomendado = ORDEN_DIAS[(idx_fuerte - 2) % 7]
        print(f"  Para llegar con tracción al día fuerte '{dia_fuerte}': tener el vídeo listo/publicado "
              f"para el '{dia_recomendado}' (2 días antes).")


Opinión post-partido (Short): día actual más usado para publicar -> Tuesday
  Para llegar con tracción al día fuerte 'Saturday': tener el vídeo listo/publicado para el 'Thursday' (2 días antes).
  Para llegar con tracción al día fuerte 'Sunday': tener el vídeo listo/publicado para el 'Friday' (2 días antes).

Seguimiento del club (Largo): día actual más usado para publicar -> Friday
  Para llegar con tracción al día fuerte 'Saturday': tener el vídeo listo/publicado para el 'Thursday' (2 días antes).
  Para llegar con tracción al día fuerte 'Sunday': tener el vídeo listo/publicado para el 'Friday' (2 días antes).


## 3. Shorts de 'Opinión post-partido' vs. otras categorías (Shorts) — día fuerte vs. resto

In [10]:
shorts = video[video['formato'] == 'Short'].copy()
shorts['es_opinion'] = shorts['categoria'] == 'Opinión post-partido'

comparativa_opinion = shorts.groupby(['es_opinion', 'es_dia_fuerte'])[
    ['views_totales', 'likes', 'ratio_likes_vista', 'retencion_media']
].mean()
comparativa_opinion.index = comparativa_opinion.index.set_names(['Es Opinión', 'Día fuerte'])
print('Shorts — Opinión post-partido vs. resto, cruzado con día fuerte / resto de la semana:')
display(comparativa_opinion.round(3))

Shorts — Opinión post-partido vs. resto, cruzado con día fuerte / resto de la semana:


views_totales    likes  ratio_likes_vista  \
Es Opinión Día fuerte                                              
False      False             6688.22  252.659              0.037   
           True              5800.50  209.100              0.038   
True       False             8844.00  321.353              0.037   
           True              5723.60  182.800              0.034   

                       retencion_media  
Es Opinión Día fuerte                   
False      False                 0.379  
           True                  0.385  
True       False                 0.407  
           True                  0.395

In [11]:
# Lecturas explícitas de las 4 celdas del cruce
try:
    op_fuerte = comparativa_opinion.loc[(True, True)]
    op_resto = comparativa_opinion.loc[(True, False)]
    otras_fuerte = comparativa_opinion.loc[(False, True)]
    otras_resto = comparativa_opinion.loc[(False, False)]

    print(f"Opinión (Short) en día fuerte vs. resto de la semana: "
          f"{op_fuerte['views_totales']:.0f} vs. {op_resto['views_totales']:.0f} vistas medias "
          f"({'MEJOR' if op_fuerte['views_totales'] > op_resto['views_totales'] else 'PEOR'} en día fuerte)")
    print(f"Opinión (Short) vs. otras categorías (Short), en día fuerte: "
          f"{op_fuerte['views_totales']:.0f} vs. {otras_fuerte['views_totales']:.0f} vistas medias "
          f"({'Opinión SUPERA' if op_fuerte['views_totales'] > otras_fuerte['views_totales'] else 'Opinión NO supera'} al resto)")
except KeyError as e:
    print(f'Alguna combinación no tiene datos suficientes: {e}')

Opinión (Short) en día fuerte vs. resto de la semana: 5724 vs. 8844 vistas medias (PEOR en día fuerte)
Opinión (Short) vs. otras categorías (Short), en día fuerte: 5724 vs. 5800 vistas medias (Opinión NO supera al resto)


## 4. Vídeos largos de 'Seguimiento del club' vs. otras categorías (Largos) — día fuerte vs. resto

In [12]:
largos = video[video['formato'] == 'Largo'].copy()
largos['es_seguimiento'] = largos['categoria'] == 'Seguimiento del club'

comparativa_seguimiento = largos.groupby(['es_seguimiento', 'es_dia_fuerte'])[
    ['views_totales', 'likes', 'ratio_likes_vista', 'retencion_media']
].mean()
comparativa_seguimiento.index = comparativa_seguimiento.index.set_names(['Es Seguimiento', 'Día fuerte'])
print('Largos — Seguimiento del club vs. resto, cruzado con día fuerte / resto de la semana:')
display(comparativa_seguimiento.round(3))

Largos — Seguimiento del club vs. resto, cruzado con día fuerte / resto de la semana:


views_totales    likes  ratio_likes_vista  \
Es Seguimiento Día fuerte                                              
False          False            4538.830  173.890              0.037   
               True             4233.286  166.629              0.039   
True           False            6091.786  227.238              0.038   
               True             5275.750  192.417              0.037   

                           retencion_media  
Es Seguimiento Día fuerte                   
False          False                 0.390  
               True                  0.386  
True           False                 0.390  
               True                  0.393

In [13]:
try:
    seg_fuerte = comparativa_seguimiento.loc[(True, True)]
    seg_resto = comparativa_seguimiento.loc[(True, False)]
    otras_fuerte_l = comparativa_seguimiento.loc[(False, True)]

    print(f"Seguimiento (Largo) en día fuerte vs. resto de la semana: "
          f"{seg_fuerte['views_totales']:.0f} vs. {seg_resto['views_totales']:.0f} vistas medias "
          f"({'MEJOR' if seg_fuerte['views_totales'] > seg_resto['views_totales'] else 'PEOR'} en día fuerte)")
    print(f"Seguimiento (Largo) vs. otras categorías (Largo), en día fuerte: "
          f"{seg_fuerte['views_totales']:.0f} vs. {otras_fuerte_l['views_totales']:.0f} vistas medias "
          f"({'Seguimiento SUPERA' if seg_fuerte['views_totales'] > otras_fuerte_l['views_totales'] else 'Seguimiento NO supera'} al resto)")
except KeyError as e:
    print(f'Alguna combinación no tiene datos suficientes: {e}')

Seguimiento (Largo) en día fuerte vs. resto de la semana: 5276 vs. 6092 vistas medias (PEOR en día fuerte)
Seguimiento (Largo) vs. otras categorías (Largo), en día fuerte: 5276 vs. 4233 vistas medias (Seguimiento SUPERA al resto)


## 5. Vídeos virales: Shorts vs. Largo, comparados con no-virales de su MISMA categoría

Se listan todos los vídeos con `z_views_categoria_formato > 2` (en tu ejecución deberían ser 18).

In [14]:
picos_virales = video[video['z_views_categoria_formato'] > 2]
print(f'Vídeos virales detectados: {len(picos_virales)}')
display(picos_virales['formato'].value_counts())

columnas_virales = ['titulo', 'categoria', 'formato', 'fecha_publicacion', 'dia_semana_publicacion',
                     'antiguedad_dias', 'views_totales', 'views_por_dia', 'retencion_media']
display(picos_virales[columnas_virales].sort_values('views_por_dia', ascending=False).round(2))

Vídeos virales detectados: 18


formato
Largo    17
Short     1
Name: count, dtype: int64

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10960\3508713555.py:7: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(picos_virales[columnas_virales].sort_values('views_por_dia', ascending=False).round(2))


,titulo,categoria,formato,fecha_publicacion,dia_semana_publicacion,antiguedad_dias,views_totales,views_por_dia,retencion_media
336,Nuestra opinión tras el partido #337,Opinión post-partido,Largo,2026-08-29,Saturday,16,14242,890.12,0.34
274,Nuestra opinión tras el partido #275,Opinión post-partido,Largo,2024-07-24,Wednesday,782,17758,22.71,0.35
258,Rumores de fichajes: novedad #259,Fichajes,Largo,2024-01-09,Tuesday,979,17967,18.35,0.40
260,Análisis del partido - Jornada 261,Seguimiento del club,Largo,2024-02-08,Thursday,949,16776,17.68,0.35
244,Rumores de fichajes: novedad #245,Fichajes,Largo,2023-07-20,Thursday,1152,17030,14.78,0.39
209,Análisis del partido - Jornada 210,Seguimiento del club,Largo,2022-05-20,Friday,1578,17716,11.23,0.39
198,Análisis del partido - Jornada 199,Seguimiento del club,Largo,2022-01-04,Tuesday,1714,18378,10.72,0.42
168,Análisis del partido - Jornada 169,Seguimiento del club,Largo,2020-12-24,Thursday,2090,19849,9.50,0.43
173,Análisis del partido - Jornada 174,Seguimiento del club,Largo,2021-02-28,Sunday,2024,18388,9.08,0.37
142,Análisis del partido - Jornada 143,Seguimiento del club,Largo,2020-02-07,Friday,2411,18313,7.60,0.37


In [15]:
for formato in ['Short', 'Largo']:
    virales_f = picos_virales[picos_virales['formato'] == formato]
    if len(virales_f) == 0:
        print(f'\nSin virales de tipo {formato}.')
        continue
    print(f"\n--- {formato}s virales (n={len(virales_f)}) vs. no-virales de SU MISMA categoría ---")
    for categoria in virales_f['categoria'].unique():
        viral_cat = virales_f[virales_f['categoria'] == categoria]
        no_viral_cat = video[
            (video['categoria'] == categoria) & (video['formato'] == formato)
            & (video['z_views_categoria_formato'] <= 2)
        ]
        # Mediana en vez de media: views_por_dia es un ratio con denominador pequeño para vídeos
        # recientes, así que unos pocos casos extremos pueden desvirtuar la media por completo.
        print(f"  {categoria}: viral n={len(viral_cat)} | views_por_dia (mediana) "
              f"{viral_cat['views_por_dia'].median():.1f} vs. no-viral {no_viral_cat['views_por_dia'].median():.1f} | "
              f"retención {viral_cat['retencion_media'].mean()*100:.1f}% vs. {no_viral_cat['retencion_media'].mean()*100:.1f}%")


--- Shorts virales (n=1) vs. no-virales de SU MISMA categoría ---
  Seguimiento del club: viral n=1 | views_por_dia (mediana) 6.4 vs. no-viral 2.1 | retención 36.7% vs. 38.3%

--- Largos virales (n=17) vs. no-virales de SU MISMA categoría ---
  Seguimiento del club: viral n=8 | views_por_dia (mediana) 9.3 vs. no-viral 2.3 | retención 39.1% vs. 39.1%
  Opinión post-partido: viral n=5 | views_por_dia (mediana) 6.1 vs. no-viral 1.0 | retención 38.0% vs. 39.0%
  Afición: viral n=1 | views_por_dia (mediana) 5.7 vs. no-viral 0.9 | retención 41.7% vs. 37.9%
  Fichajes: viral n=3 | views_por_dia (mediana) 14.8 vs. no-viral 1.5 | retención 38.7% vs. 38.9%


## 6. Diagnóstico de 'Afición': ¿contenido débil o factor externo?

Se descartan explícitamente: tamaño de muestra, mezcla de formato, y antigüedad/novedad antes de concluir nada sobre la calidad del contenido en sí.

In [16]:
conteo_categorias = video['categoria'].value_counts()
print('Vídeos por categoría:')
display(conteo_categorias)

n_aficion = conteo_categorias.get('Afición', 0)
n_media_resto = conteo_categorias.drop('Afición', errors='ignore').mean()
print(f"\nAfición tiene {n_aficion} vídeos; el resto de categorías tiene de media {n_media_resto:.0f}.")
if n_aficion < n_media_resto * 0.5:
    print('AVISO: Afición tiene bastante menos de la mitad de vídeos que la categoría media — cualquier '
          'diferencia de rendimiento debe leerse con cautela por tamaño de muestra.')

Vídeos por categoría:


categoria
Seguimiento del club      154
Opinión post-partido       90
Fichajes                   54
Curiosidades de fútbol     23
Afición                    17
Name: count, dtype: int64


Afición tiene 17 vídeos; el resto de categorías tiene de media 80.
AVISO: Afición tiene bastante menos de la mitad de vídeos que la categoría media — cualquier diferencia de rendimiento debe leerse con cautela por tamaño de muestra.


In [17]:
aficion = video[video['categoria'] == 'Afición']
resto = video[video['categoria'] != 'Afición']

# Descarte 1: mezcla de formato distinta
mix_aficion = aficion['formato'].value_counts(normalize=True).mul(100)
mix_resto = resto['formato'].value_counts(normalize=True).mul(100)
print('Mezcla de formato — Afición vs. resto del canal:')
display(pd.DataFrame({'Afición': mix_aficion, 'Resto': mix_resto}).round(1))

# Descarte 2: antigüedad / novedad
print(f"\nAntigüedad media — Afición: {aficion['antiguedad_dias'].mean():.0f} días | "
      f"Resto: {resto['antiguedad_dias'].mean():.0f} días")

Mezcla de formato — Afición vs. resto del canal:


,Afición,Resto
formato,,
Largo,76.5,75.4
Short,23.5,24.6



Antigüedad media — Afición: 2015 días | Resto: 2085 días


In [18]:
# Comparación controlada: usando la MEDIANA de views_por_dia (más robusta que la media frente a
# vídeos muy recientes con denominador casi cero), y separado por formato
# (neutraliza el efecto de mezcla Shorts/largos). Se excluyen además los vídeos con menos de 7 días
# de antigüedad (apto_velocidad), que no han tenido tiempo de alcanzar una velocidad estable.
print('Comparación controlada por formato y velocidad (mediana de views_por_dia, no vistas totales crudas):')
for formato in ['Short', 'Largo']:
    a = aficion[(aficion['formato'] == formato) & aficion['apto_velocidad']]
    r = resto[(resto['formato'] == formato) & resto['apto_velocidad']]
    if len(a) == 0 or len(r) == 0:
        print(f'  {formato}: datos insuficientes en Afición o en el resto para comparar.')
        continue
    print(f"  {formato} — Afición (n={len(a)}): {a['views_por_dia'].median():.2f} vistas/día (mediana), "
          f"retención {a['retencion_media'].mean()*100:.1f}% | "
          f"Resto (n={len(r)}): {r['views_por_dia'].median():.2f} vistas/día (mediana), "
          f"retención {r['retencion_media'].mean()*100:.1f}%")

Comparación controlada por formato y velocidad (mediana de views_por_dia, no vistas totales crudas):
  Short — Afición (n=4): 1.16 vistas/día (mediana), retención 35.6% | Resto (n=79): 2.24 vistas/día (mediana), retención 38.9%
  Largo — Afición (n=13): 1.18 vistas/día (mediana), retención 38.2% | Resto (n=241): 1.98 vistas/día (mediana), retención 39.1%


In [19]:
# Veredicto final, tras los descartes anteriores — con mediana y filtro de antigüedad mínima
aficion_apta = aficion[aficion['apto_velocidad']]
resto_apto = resto[resto['apto_velocidad']]
gap_velocidad = aficion_apta['views_por_dia'].median() / resto_apto['views_por_dia'].median() - 1
gap_retencion = aficion['retencion_media'].mean() - resto['retencion_media'].mean()

print(f"Gap de velocidad (mediana, Afición vs. resto, ya controlado por antigüedad): {gap_velocidad*100:+.1f}%")
print(f"Gap de retención: {gap_retencion*100:+.1f} puntos porcentuales")

if n_aficion < n_media_resto * 0.5:
    print('\nVEREDICTO: con este tamaño de muestra, cualquier conclusión sobre \'el contenido en sí\' es '
          'poco fiable — antes de decidir nada, GrowUP necesitaría más vídeos de Afición para confirmar el '
          'patrón, no solo estos pocos.')
elif gap_velocidad < -0.15 and gap_retencion < -0.03:
    print('\nVEREDICTO: incluso controlando por formato, antigüedad y usando la mediana (robusta a outliers), '
          'Afición sigue por debajo en velocidad Y retención — el patrón apunta al contenido en sí, no a un '
          'artefacto de los datos.')
else:
    print('\nVEREDICTO: la diferencia se explica en buena parte por formato/antigüedad/outliers puntuales, '
          'no por el contenido en sí — no hay evidencia sólida de que Afición \'funcione peor\' de forma inherente.')

Gap de velocidad (mediana, Afición vs. resto, ya controlado por antigüedad): -43.4%
Gap de retención: -1.4 puntos porcentuales

VEREDICTO: con este tamaño de muestra, cualquier conclusión sobre 'el contenido en sí' es poco fiable — antes de decidir nada, GrowUP necesitaría más vídeos de Afición para confirmar el patrón, no solo estos pocos.


## 7. Vídeos con vistas anormalmente altas: ¿en qué clasificaciones caen, y qué pasó con los suscriptores?

In [20]:
columnas_cruce = ['titulo', 'categoria', 'formato', 'fecha_publicacion', 'dia_semana_publicacion', 'nivel_views']
print('Vídeos con vistas anormalmente altas (mismos del punto 5) — clasificación del punto 1:')
display(picos_virales[columnas_cruce])

top10_views_ids = set(video.sort_values('views_totales', ascending=False).head(10)['video_id'])
top10_vel_ids = set(video[video['apto_velocidad']].sort_values('views_por_dia', ascending=False).head(10)['video_id'])
top10_retencion_ids = set(video.sort_values('retencion_media', ascending=False).head(10)['video_id'])

for _, v in picos_virales.iterrows():
    etiquetas = []
    if v['video_id'] in top10_views_ids: etiquetas.append('Top10 vistas totales')
    if v['video_id'] in top10_vel_ids: etiquetas.append('Top10 velocidad')
    if v['video_id'] in top10_retencion_ids: etiquetas.append('Top10 retención')
    print(f"  {v['titulo'][:40]:40s} -> {', '.join(etiquetas) if etiquetas else '(ninguna clasificación Top10)'}")

Vídeos con vistas anormalmente altas (mismos del punto 5) — clasificación del punto 1:


,titulo,categoria,formato,fecha_publicacion,dia_semana_publicacion,nivel_views
45,Análisis del partido - Jornada 46,Seguimiento del club,Largo,2016-10-27,Thursday,Muy alto
54,Análisis del partido - Jornada 55,Seguimiento del club,Largo,2017-02-19,Sunday,Muy alto
59,Nuestra opinión tras el partido #60,Opinión post-partido,Largo,2017-04-19,Wednesday,Muy alto
84,Análisis del partido - Jornada 85,Seguimiento del club,Short,2018-02-22,Thursday,Muy alto
106,Así vive la afición el partido #107,Afición,Largo,2018-11-19,Monday,Muy alto
119,Nuestra opinión tras el partido #120,Opinión post-partido,Largo,2019-04-28,Sunday,Muy alto
139,Nuestra opinión tras el partido #140,Opinión post-partido,Largo,2019-12-31,Tuesday,Muy alto
142,Análisis del partido - Jornada 143,Seguimiento del club,Largo,2020-02-07,Friday,Muy alto
145,Rumores de fichajes: novedad #146,Fichajes,Largo,2020-03-15,Sunday,Muy alto
168,Análisis del partido - Jornada 169,Seguimiento del club,Largo,2020-12-24,Thursday,Muy alto


  Análisis del partido - Jornada 46        -> (ninguna clasificación Top10)
  Análisis del partido - Jornada 55        -> Top10 vistas totales
  Nuestra opinión tras el partido #60      -> (ninguna clasificación Top10)
  Análisis del partido - Jornada 85        -> Top10 vistas totales
  Así vive la afición el partido #107      -> (ninguna clasificación Top10)
  Nuestra opinión tras el partido #120     -> (ninguna clasificación Top10)
  Nuestra opinión tras el partido #140     -> Top10 retención
  Análisis del partido - Jornada 143       -> Top10 vistas totales
  Rumores de fichajes: novedad #146        -> (ninguna clasificación Top10)
  Análisis del partido - Jornada 169       -> Top10 vistas totales
  Análisis del partido - Jornada 174       -> Top10 vistas totales
  Análisis del partido - Jornada 199       -> Top10 vistas totales
  Análisis del partido - Jornada 210       -> (ninguna clasificación Top10)
  Rumores de fichajes: novedad #245        -> (ninguna clasificación Top10)
  Ru

In [21]:
# Cruce con ganancia neta de suscriptores: desde el día de publicación hasta 3 días después
# Solo es posible para vídeos cuya fecha de publicación cae dentro de la ventana de evolucion_diaria (30 días)
fecha_min_evolucion = evolucion['fecha'].min()
fecha_max_evolucion = evolucion['fecha'].max()

resultados_cruce = []
for _, v in picos_virales.iterrows():
    fecha_pub = v['fecha_publicacion'].normalize()
    if fecha_pub < fecha_min_evolucion or fecha_pub > fecha_max_evolucion:
        resultados_cruce.append({
            'titulo': v['titulo'], 'fecha_publicacion': fecha_pub.date(),
            'dia_semana_publicacion': v['dia_semana_publicacion'],
            'suscriptores_netos_3dias': None, 'nota': 'fuera de la ventana de 30 días de evolucion_diaria',
        })
        continue
    ventana = evolucion[
        (evolucion['fecha'] >= fecha_pub) & (evolucion['fecha'] <= fecha_pub + pd.Timedelta(days=3))
    ]
    resultados_cruce.append({
        'titulo': v['titulo'], 'fecha_publicacion': fecha_pub.date(),
        'dia_semana_publicacion': v['dia_semana_publicacion'],
        'suscriptores_netos_3dias': int(ventana['suscriptores_netos'].sum()) if len(ventana) else None,
        'nota': f'{len(ventana)} día(s) disponibles en la ventana' if len(ventana) else 'sin datos en la ventana',
    })

df_cruce = pd.DataFrame(resultados_cruce)
display(df_cruce)

con_dato = df_cruce[df_cruce['suscriptores_netos_3dias'].notna()]
if len(con_dato):
    media_normal = evolucion['suscriptores_netos'].mean() * 4  # equivalente a 4 días de referencia (día pub + 3 después)
    print(f"\nMedia de suscriptores netos en una ventana de 4 días cualquiera (referencia): {media_normal:.1f}")
    print(f"Media real tras estos vídeos virales: {con_dato['suscriptores_netos_3dias'].mean():.1f}")
else:
    print('\nNinguno de los vídeos virales cae dentro de la ventana de 30 días de evolucion_diaria — '
          'no se puede hacer el cruce con datos disponibles actualmente. Sería necesario ampliar el rango '
          'de evolucion_diaria (Fase 4) para poder analizar el impacto en suscriptores de vídeos antiguos.')

,titulo,fecha_publicacion,dia_semana_publicacion,suscriptores_netos_3dias,nota
0,Análisis del partido - Jornada 46,2016-10-27,Thursday,NaN,fuera de la ventana de 30 días de evolucion_di...
1,Análisis del partido - Jornada 55,2017-02-19,Sunday,NaN,fuera de la ventana de 30 días de evolucion_di...
2,Nuestra opinión tras el partido #60,2017-04-19,Wednesday,NaN,fuera de la ventana de 30 días de evolucion_di...
3,Análisis del partido - Jornada 85,2018-02-22,Thursday,NaN,fuera de la ventana de 30 días de evolucion_di...
4,Así vive la afición el partido #107,2018-11-19,Monday,NaN,fuera de la ventana de 30 días de evolucion_di...
5,Nuestra opinión tras el partido #120,2019-04-28,Sunday,NaN,fuera de la ventana de 30 días de evolucion_di...
6,Nuestra opinión tras el partido #140,2019-12-31,Tuesday,NaN,fuera de la ventana de 30 días de evolucion_di...
7,Análisis del partido - Jornada 143,2020-02-07,Friday,NaN,fuera de la ventana de 30 días de evolucion_di...
8,Rumores de fichajes: novedad #146,2020-03-15,Sunday,NaN,fuera de la ventana de 30 días de evolucion_di...
9,Análisis del partido - Jornada 169,2020-12-24,Thursday,NaN,fuera de la ventana de 30 días de evolucion_di...



Media de suscriptores netos en una ventana de 4 días cualquiera (referencia): 104.9
Media real tras estos vídeos virales: 115.0
